In [1]:
# ================================================================
# N-CMAPSS DS02-006
# CNN + Transformer Dual Branch + Gated Fusion
#
# Input:
#   18 raw sensor features
#
# Branch 1:
#   CNN -> Local Feature
#
# Branch 2:
#   Transformer -> Global Feature
#
# Fusion:
#   Gated Fusion
#
# NO Physics Constraint
# NO Attention Pooling
# ================================================================

import os
import time
import h5py
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ================================================================
# 1. CONFIG
# ================================================================

DATA_PATH = r"E:\RUL\N-CMAPSS_DS02-006.h5"

CHECKPOINT_PATH = (
    r"E:\RUL\best_cnn_transformer_gated_ds02_006.pth"
)

WINDOW = 50
STRIDE = 5

BATCH_SIZE = 256

INPUT_DIM = 18

CNN_DIM = 96
D_MODEL = 96

NHEAD = 8
NUM_LAYERS = 2

DROPOUT = 0.15

EPOCHS = 20

LR = 3e-4

WEIGHT_DECAY = 1e-4

PATIENCE = 5

NUM_WORKERS = 0


# ================================================================
# DEVICE
# ================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = (
    DEVICE.type == "cuda"
)


# ================================================================
# PRINT CONFIG
# ================================================================

print("=" * 75)

print(
    "N-CMAPSS DS02-006 | "
    "CNN + TRANSFORMER DUAL BRANCH + GATED FUSION"
)

print("=" * 75)

print("Device       :", DEVICE)

if DEVICE.type == "cuda":

    print(
        "GPU          :",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory   :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

print("AMP          :", AMP_ENABLED)

print("Window       :", WINDOW)

print("Stride       :", STRIDE)

print("Batch size   :", BATCH_SIZE)

print("Input dim    :", INPUT_DIM)

print("CNN dim      :", CNN_DIM)

print("d_model      :", D_MODEL)

print("Transformer  :", NUM_LAYERS, "layers")

print("Learning rate:", LR)


# ================================================================
# 2. POSITIONAL ENCODING
# ================================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=512
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            *
            (
                -np.log(10000.0)
                / d_model
            )
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )


    def forward(self, x):

        return (
            x
            +
            self.pe[
                :, :x.size(1)
            ]
        )


# ================================================================
# 3. WINDOW DATASET
# ================================================================

class WindowDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        units
    ):

        self.X = torch.from_numpy(
            X.astype(np.float32)
        )

        self.Y = torch.from_numpy(
            Y.astype(np.float32)
        )

        self.indices = []


        unique_units = np.unique(
            units
        )


        for unit in unique_units:

            idx = np.where(
                units == unit
            )[0]

            n = len(idx)


            if n < WINDOW:

                continue


            starts = list(
                idx[
                    :n - WINDOW + 1:
                    STRIDE
                ]
            )


            last_start = (
                idx[-1]
                - WINDOW
                + 1
            )


            if (
                len(starts) == 0
                or
                starts[-1]
                != last_start
            ):

                starts.append(
                    last_start
                )


            self.indices.extend(
                starts
            )


    def __len__(self):

        return len(
            self.indices
        )


    def __getitem__(
        self,
        index
    ):

        start = (
            self.indices[index]
        )

        end = (
            start
            + WINDOW
        )


        x = self.X[
            start:end
        ]


        # 使用窗口最后一个时间点
        # 对应的 RUL
        y = self.Y[
            end - 1
        ]


        return x, y


# ================================================================
# 4. DUAL BRANCH MODEL
# ================================================================

class DualBranchCNNTransformer(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        # ========================================================
        # BRANCH 1
        # CNN LOCAL FEATURE EXTRACTOR
        # ========================================================

        self.cnn_branch = nn.Sequential(

            nn.Conv1d(
                INPUT_DIM,
                64,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(64),

            nn.GELU(),

            nn.Dropout(DROPOUT),


            nn.Conv1d(
                64,
                CNN_DIM,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(
                CNN_DIM
            ),

            nn.GELU()
        )


        # ========================================================
        # BRANCH 2
        # TRANSFORMER GLOBAL FEATURE EXTRACTOR
        #
        # 注意：
        # Transformer直接使用18维原始传感器
        # 经过 Linear Projection -> d_model
        # ========================================================

        self.transformer_projection = nn.Linear(
            INPUT_DIM,
            D_MODEL
        )


        self.pos_encoder = (
            PositionalEncoding(
                D_MODEL
            )
        )


        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=D_MODEL,

                nhead=NHEAD,

                dim_feedforward=
                    D_MODEL * 4,

                dropout=DROPOUT,

                activation="gelu",

                batch_first=True,

                norm_first=True
            )
        )


        self.transformer_branch = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=NUM_LAYERS
            )
        )


        # ========================================================
        # BRANCH POOLING
        # ========================================================

        # CNN:
        # [B, 50, 96]
        #
        # Transformer:
        # [B, 50, 96]
        #
        # 使用 Mean Pooling
        # 保证实验简单、稳定
        # ========================================================

        self.cnn_pool = nn.AdaptiveAvgPool1d(1)


        # ========================================================
        # GATED FUSION
        # ========================================================

        # CNN feature:
        # [B, 96]
        #
        # Transformer feature:
        # [B, 96]
        #
        # concat:
        # [B, 192]
        #
        # gate:
        # [B, 96]
        # ========================================================

        self.gate = nn.Sequential(

            nn.Linear(
                CNN_DIM + D_MODEL,
                D_MODEL
            ),

            nn.GELU(),

            nn.Linear(
                D_MODEL,
                D_MODEL
            ),

            nn.Sigmoid()
        )


        # ========================================================
        # FUSION PROJECTION
        # ========================================================

        self.fusion = nn.Sequential(

            nn.Linear(
                CNN_DIM + D_MODEL,
                D_MODEL
            ),

            nn.LayerNorm(
                D_MODEL
            ),

            nn.GELU(),

            nn.Dropout(
                DROPOUT
            )
        )


        # ========================================================
        # REGRESSOR
        # ========================================================

        self.regressor = nn.Sequential(

            nn.Linear(
                D_MODEL,
                64
            ),

            nn.GELU(),

            nn.Dropout(
                DROPOUT
            ),

            nn.Linear(
                64,
                1
            )
        )


    def forward(
        self,
        x,
        return_gate=False
    ):

        # ========================================================
        # x
        #
        # [B, 50, 18]
        # ========================================================


        # ========================================================
        # CNN BRANCH
        # ========================================================

        cnn_x = x.transpose(
            1,
            2
        )

        # [B, 18, 50]


        cnn_x = self.cnn_branch(
            cnn_x
        )

        # [B, 96, 50]


        cnn_feature = self.cnn_pool(
            cnn_x
        )

        # [B, 96, 1]


        cnn_feature = (
            cnn_feature
            .squeeze(-1)
        )

        # [B, 96]


        # ========================================================
        # TRANSFORMER BRANCH
        # ========================================================

        transformer_x = (
            self.transformer_projection(
                x
            )
        )

        # [B, 50, 96]


        transformer_x = (
            self.pos_encoder(
                transformer_x
            )
        )


        transformer_x = (
            self.transformer_branch(
                transformer_x
            )
        )

        # [B, 50, 96]


        transformer_feature = (
            transformer_x.mean(
                dim=1
            )
        )

        # [B, 96]


        # ========================================================
        # CONCAT
        # ========================================================

        combined = torch.cat(
            [
                cnn_feature,
                transformer_feature
            ],
            dim=1
        )

        # [B, 192]


        # ========================================================
        # GATE
        # ========================================================

        gate_weight = self.gate(
            combined
        )

        # [B, 96]


        # ========================================================
        # GATED FEATURE FUSION
        # ========================================================

        # 将CNN和Transformer拼接后的信息
        # 压缩到96维

        fused_base = self.fusion(
            combined
        )

        # [B, 96]


        # gate控制融合特征

        fused = (
            gate_weight
            * fused_base
        )

        # [B, 96]


        # ========================================================
        # REGRESSION
        # ========================================================

        output = self.regressor(
            fused
        )

        output = output.squeeze(
            -1
        )


        # RUL不能小于0

        output = F.softplus(
            output
        )


        if return_gate:

            return (
                output,
                gate_weight
            )


        return output


# ================================================================
# 5. LOAD DATA
# ================================================================

print("\nLoading N-CMAPSS...")

if not os.path.exists(
    DATA_PATH
):

    raise FileNotFoundError(
        f"\n找不到数据文件:\n"
        f"{DATA_PATH}\n"
    )


with h5py.File(
    DATA_PATH,
    "r"
) as f:

    print(
        "Available datasets:"
    )

    print(
        list(f.keys())
    )


    A_dev = f[
        "A_dev"
    ][:]

    W_dev = f[
        "W_dev"
    ][:]

    Xs_dev = f[
        "X_s_dev"
    ][:]

    Y_dev = f[
        "Y_dev"
    ][:]


    A_test = f[
        "A_test"
    ][:]

    W_test = f[
        "W_test"
    ][:]

    Xs_test = f[
        "X_s_test"
    ][:]

    Y_test = f[
        "Y_test"
    ][:]


# ================================================================
# 6. BUILD 18-D FEATURES
# ================================================================

X_dev = np.concatenate(
    [
        W_dev,
        Xs_dev
    ],
    axis=1
).astype(
    np.float32
)


X_test = np.concatenate(
    [
        W_test,
        Xs_test
    ],
    axis=1
).astype(
    np.float32
)


Y_dev = (
    Y_dev
    .reshape(-1)
    .astype(np.float32)
)


Y_test = (
    Y_test
    .reshape(-1)
    .astype(np.float32)
)


units_dev = (
    A_dev[:, 0]
    .astype(int)
)


units_test = (
    A_test[:, 0]
    .astype(int)
)


print(
    "\nRaw dev shape :",
    X_dev.shape
)

print(
    "Raw test shape:",
    X_test.shape
)


if X_dev.shape[1] != 18:

    raise ValueError(
        f"输入维度错误: "
        f"{X_dev.shape}"
    )


# ================================================================
# 7. TRAIN / VALIDATION SPLIT
# ================================================================

TRAIN_UNITS = [
    2,
    5,
    10,
    16,
    18
]

VAL_UNIT = 20


train_mask = np.isin(
    units_dev,
    TRAIN_UNITS
)


val_mask = (
    units_dev
    == VAL_UNIT
)


X_train = X_dev[
    train_mask
]

Y_train = Y_dev[
    train_mask
]

U_train = units_dev[
    train_mask
]


X_val = X_dev[
    val_mask
]

Y_val = Y_dev[
    val_mask
]

U_val = units_dev[
    val_mask
]


print(
    "\nTrain units:",
    TRAIN_UNITS
)

print(
    "Validation unit:",
    VAL_UNIT
)


# ================================================================
# 8. NORMALIZATION
# ================================================================

print(
    "\nNormalizing..."
)


mean = X_train.mean(
    axis=0
)


std = X_train.std(
    axis=0
)


std[
    std < 1e-6
] = 1.0


X_train = (
    X_train - mean
) / std


X_val = (
    X_val - mean
) / std


X_test = (
    X_test - mean
) / std


# ================================================================
# 9. DATASET
# ================================================================

train_dataset = WindowDataset(
    X_train,
    Y_train,
    U_train
)


val_dataset = WindowDataset(
    X_val,
    Y_val,
    U_val
)


test_dataset = WindowDataset(
    X_test,
    Y_test,
    units_test
)


print(
    "\nTrain windows:",
    f"{len(train_dataset):,}"
)

print(
    "Val windows  :",
    f"{len(val_dataset):,}"
)

print(
    "Test windows :",
    f"{len(test_dataset):,}"
)


# ================================================================
# 10. DATALOADER
# ================================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=True
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True
)


print(
    "Train batches/epoch:",
    len(train_loader)
)


# ================================================================
# 11. MODEL
# ================================================================

model = (
    DualBranchCNNTransformer()
    .to(DEVICE)
)


params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "\nTrainable parameters:",
    f"{params:,}"
)


# ================================================================
# 12. LOSS
# ================================================================

criterion = nn.MSELoss()


# ================================================================
# 13. OPTIMIZER
# ================================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ================================================================
# 14. LR SCHEDULER
# ================================================================

scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="min",

        factor=0.5,

        patience=2,

        min_lr=1e-6
    )
)


# ================================================================
# 15. AMP
# ================================================================

if AMP_ENABLED:

    scaler = torch.amp.GradScaler(
        "cuda"
    )

else:

    scaler = None


# ================================================================
# 16. EVALUATION
# ================================================================

def evaluate(
    model,
    loader
):

    model.eval()


    predictions = []

    targets = []


    total_loss = 0.0

    total_n = 0


    with torch.no_grad():

        for x, y in loader:

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            y = y.to(
                DEVICE,
                non_blocking=True
            )


            if AMP_ENABLED:

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    pred = model(x)

                    loss = criterion(
                        pred,
                        y
                    )

            else:

                pred = model(x)

                loss = criterion(
                    pred,
                    y
                )


            n = y.size(0)


            total_loss += (
                loss.item()
                * n
            )

            total_n += n


            predictions.append(
                pred.float()
                .cpu()
                .numpy()
            )


            targets.append(
                y.float()
                .cpu()
                .numpy()
            )


    predictions = np.concatenate(
        predictions
    )


    targets = np.concatenate(
        targets
    )


    mse = np.mean(
        (
            predictions
            - targets
        ) ** 2
    )


    rmse = np.sqrt(
        mse
    )


    mae = np.mean(
        np.abs(
            predictions
            - targets
        )
    )


    return (
        total_loss / total_n,
        rmse,
        mae
    )


# ================================================================
# 17. TRAIN
# ================================================================

best_rmse = float(
    "inf"
)

best_epoch = 0

patience_counter = 0


for epoch in range(
    1,
    EPOCHS + 1
):

    start_time = time.time()


    model.train()


    total_loss = 0.0

    total_n = 0


    for x, y in train_loader:

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        if AMP_ENABLED:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                pred = model(x)

                loss = criterion(
                    pred,
                    y
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )


            scaler.step(
                optimizer
            )


            scaler.update()


        else:

            pred = model(x)

            loss = criterion(
                pred,
                y
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )


            optimizer.step()


        n = y.size(0)


        total_loss += (
            loss.item()
            * n
        )


        total_n += n


    train_mse = (
        total_loss
        / total_n
    )


    # ============================================================
    # VALIDATION
    # ============================================================

    val_loss, val_rmse, val_mae = (
        evaluate(
            model,
            val_loader
        )
    )


    scheduler.step(
        val_rmse
    )


    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    elapsed = (
        time.time()
        - start_time
    )


    print("\n" + "-" * 75)

    print(
        f"Epoch {epoch:02d}/{EPOCHS}"
    )

    print(
        f"Train MSE : {train_mse:.6f}"
    )

    print(
        f"Val RMSE  : {val_rmse:.6f}"
    )

    print(
        f"Val MAE   : {val_mae:.6f}"
    )

    print(
        f"LR        : {current_lr:.2e}"
    )

    print(
        f"Time      : "
        f"{elapsed:.1f}s"
    )


    # ============================================================
    # SAVE BEST
    # ============================================================

    if val_rmse < best_rmse:

        best_rmse = val_rmse

        best_epoch = epoch

        patience_counter = 0


        torch.save(

            {
                "model_state_dict":
                    model.state_dict(),

                "mean":
                    mean,

                "std":
                    std,

                "input_dim":
                    INPUT_DIM,

                "cnn_dim":
                    CNN_DIM,

                "d_model":
                    D_MODEL,

                "nhead":
                    NHEAD,

                "num_layers":
                    NUM_LAYERS,

                "window":
                    WINDOW,

                "stride":
                    STRIDE,

                "best_val_rmse":
                    best_rmse,

                "best_epoch":
                    best_epoch
            },

            CHECKPOINT_PATH
        )


        print(
            "  -> Best checkpoint saved."
        )


    else:

        patience_counter += 1


    if (
        patience_counter
        >= PATIENCE
    ):

        print(
            "\nEarly stopping."
        )

        break


# ================================================================
# 18. LOAD BEST MODEL
# ================================================================

print("\n")
print("=" * 75)

print(
    "Loading best checkpoint..."
)

print("=" * 75)


checkpoint = torch.load(

    CHECKPOINT_PATH,

    map_location=DEVICE,

    weights_only=False
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


model.eval()


# ================================================================
# 19. FINAL VALIDATION
# ================================================================

_, val_rmse, val_mae = (
    evaluate(
        model,
        val_loader
    )
)


# ================================================================
# 20. FINAL TEST
# ================================================================

_, test_rmse, test_mae = (
    evaluate(
        model,
        test_loader
    )
)


# ================================================================
# 21. FINAL RESULTS
# ================================================================

print("\n")

print("=" * 75)

print(
    "FINAL RESULTS | "
    "CNN + TRANSFORMER DUAL BRANCH + GATED FUSION"
)

print("=" * 75)


print(
    f"Best Epoch : {best_epoch}"
)


print(
    f"Val RMSE   : {val_rmse:.6f}"
)


print(
    f"Val MAE    : {val_mae:.6f}"
)


print(
    f"Test RMSE  : {test_rmse:.6f}"
)


print(
    f"Test MAE   : {test_mae:.6f}"
)


print(
    "\nCheckpoint:"
)


print(
    CHECKPOINT_PATH
)


print("=" * 75)

N-CMAPSS DS02-006 | CNN + TRANSFORMER DUAL BRANCH + GATED FUSION
Device       : cuda
GPU          : NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory   : 4.0 GB
AMP          : True
Window       : 50
Stride       : 5
Batch size   : 256
Input dim    : 18
CNN dim      : 96
d_model      : 96
Transformer  : 2 layers
Learning rate: 0.0003

Loading N-CMAPSS...
Available datasets:
['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']

Raw dev shape : (5263447, 18)
Raw test shape: (1253743, 18)

Train units: [2, 5, 10, 16, 18]
Validation unit: 20

Normalizing...

Train windows: 899,014
Val windows  : 153,623
Test windows : 250,722
Train batches/epoch: 3512

Trainable parameters: 303,009


e:\RUL\venv\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



---------------------------------------------------------------------------
Epoch 01/20
Train MSE : 205.372301
Val RMSE  : 9.902736
Val MAE   : 7.619159
LR        : 3.00e-04
Time      : 189.8s
  -> Best checkpoint saved.

---------------------------------------------------------------------------
Epoch 02/20
Train MSE : 67.632730
Val RMSE  : 8.912691
Val MAE   : 7.245178
LR        : 3.00e-04
Time      : 186.0s
  -> Best checkpoint saved.

---------------------------------------------------------------------------
Epoch 03/20
Train MSE : 59.563915
Val RMSE  : 9.810393
Val MAE   : 7.539559
LR        : 3.00e-04
Time      : 182.5s

---------------------------------------------------------------------------
Epoch 04/20
Train MSE : 54.444717
Val RMSE  : 9.099660
Val MAE   : 6.851097
LR        : 3.00e-04
Time      : 189.4s

---------------------------------------------------------------------------
Epoch 05/20
Train MSE : 50.474607
Val RMSE  : 9.628892
Val MAE   : 7.450392
LR        : 1.50e-

In [ ]:
# ================================================================
# N-CMAPSS DS02-006
# CNN + Transformer Fusion V2
#
# CNN:
#   Local temporal feature extraction
#
# Transformer:
#   Global temporal dependency modeling
#
# Fusion:
#   LayerNorm
#   Channel Attention
#   Dynamic Gated Fusion
#   Residual Fusion
#
# Training:
#   Main prediction loss
#   CNN auxiliary loss
#   Transformer auxiliary loss
#
# ================================================================

import os
import time
import random
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


# ================================================================
# 1. CONFIG
# ================================================================

SEED = 42

DATA_FILE = "N-CMAPSS_DS02-006.h5"

MODEL_FILE = "best_cnn_transformer_fusion_v2_ds02_006.pth"

RESULT_CSV = "cnn_transformer_fusion_v2_test_predictions.csv"
ENGINE_CSV = "cnn_transformer_fusion_v2_engine_metrics.csv"
FIG_FILE = "cnn_transformer_fusion_v2_results.png"

WINDOW_SIZE = 50

BATCH_SIZE = 512

MAX_EPOCHS = 30

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4

PATIENCE = 7

D_MODEL = 96

NHEAD = 8

NUM_TRANSFORMER_LAYERS = 2

DROPOUT = 0.15

# 辅助损失权重
CNN_AUX_WEIGHT = 0.15
TRANSFORMER_AUX_WEIGHT = 0.15

# 是否使用 AMP
USE_AMP = torch.cuda.is_available()


# ================================================================
# 2. RANDOM SEED
# ================================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False

    torch.backends.cudnn.benchmark = True


set_seed(SEED)


# ================================================================
# 3. DEVICE
# ================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("N-CMAPSS DS02-006 | CNN + TRANSFORMER FUSION V2")
print("=" * 70)

print(f"Device       : {device}")

if torch.cuda.is_available():

    print(
        f"GPU          : "
        f"{torch.cuda.get_device_name(0)}"
    )

print(f"Window       : {WINDOW_SIZE}")

print(f"Batch        : {BATCH_SIZE}")

print(f"d_model      : {D_MODEL}")

print(f"Transformer  : {NUM_TRANSFORMER_LAYERS} layers")

print(f"AMP          : {USE_AMP}")

print("=" * 70)


# ================================================================
# 4. LOAD DATA
# ================================================================

print("\nLoading N-CMAPSS data...")

with h5py.File(DATA_FILE, "r") as f:

    W_dev = f["W_dev"][:]

    Xs_dev = f["X_s_dev"][:]

    A_dev = f["A_dev"][:]

    Y_dev = f["Y_dev"][:]

    W_test = f["W_test"][:]

    Xs_test = f["X_s_test"][:]

    A_test = f["A_test"][:]

    Y_test = f["Y_test"][:]


# ------------------------------------------------
# W = 4
# Xs = 14
# total = 18
# ------------------------------------------------

X_dev_raw = np.concatenate(
    [W_dev, Xs_dev],
    axis=1
).astype(np.float32)

X_test_raw = np.concatenate(
    [W_test, Xs_test],
    axis=1
).astype(np.float32)


unit_dev = A_dev[:, 0].astype(np.int32)

unit_test = A_test[:, 0].astype(np.int32)

Y_dev = Y_dev.reshape(-1).astype(np.float32)

Y_test = Y_test.reshape(-1).astype(np.float32)


print(f"X_dev  : {X_dev_raw.shape}")

print(f"X_test : {X_test_raw.shape}")

print(f"Y_dev  : {Y_dev.shape}")

print(f"Y_test : {Y_test.shape}")

print(
    f"Dev engines  : "
    f"{np.unique(unit_dev)}"
)

print(
    f"Test engines : "
    f"{np.unique(unit_test)}"
)


# ================================================================
# 5. TRAIN / VALIDATION ENGINE SPLIT
# ================================================================

unique_units = np.unique(unit_dev)

# 最后一台发动机作为验证
val_unit = unique_units[-1]

train_units = unique_units[:-1]


train_mask = np.isin(
    unit_dev,
    train_units
)

val_mask = (
    unit_dev == val_unit
)


print("\n" + "=" * 70)

print("ENGINE SPLIT")

print("=" * 70)

print(
    f"Train engines : "
    f"{train_units}"
)

print(
    f"Validation    : "
    f"{val_unit}"
)


# ================================================================
# 6. NORMALIZATION
# ================================================================
#
# 非常重要：
#
# 只使用 TRAIN ENGINE 计算 min/max
#
# 不能：
#
# X_dev.min()
#
# 因为 validation engine 不应该参与 scaler fitting。
#
# ================================================================

X_train_raw = X_dev_raw[train_mask]

X_min = X_train_raw.min(axis=0)

X_max = X_train_raw.max(axis=0)

denom = X_max - X_min

denom[denom < 1e-8] = 1.0


X_dev = (
    2.0 *
    (X_dev_raw - X_min)
    / denom
    - 1.0
)


X_test = (
    2.0 *
    (X_test_raw - X_min)
    / denom
    - 1.0
)


X_dev = X_dev.astype(np.float32)

X_test = X_test.astype(np.float32)


print("\nNormalization:")

print("Scaler fitted ONLY on training engines.")


# ================================================================
# 7. WINDOW DATASET
# ================================================================

class NCMAPSSDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        units,
        window_size=50
    ):

        self.X = torch.from_numpy(
            np.ascontiguousarray(X)
        )

        self.Y = torch.from_numpy(
            np.ascontiguousarray(Y)
        ).float()

        self.units = units

        self.window_size = window_size

        self.valid_indices = []

        unique_units = np.unique(units)

        for u in unique_units:

            idx = np.where(
                units == u
            )[0]

            if len(idx) < window_size:

                continue

            for i in range(
                len(idx) - window_size + 1
            ):

                self.valid_indices.append(
                    idx[i]
                )

        self.valid_indices = np.asarray(
            self.valid_indices,
            dtype=np.int64
        )


    def __len__(self):

        return len(self.valid_indices)


    def __getitem__(self, index):

        start = self.valid_indices[index]

        end = start + self.window_size

        x = self.X[start:end]

        y = self.Y[end - 1]

        return x, y


# ================================================================
# 8. DATASET
# ================================================================

train_dataset = NCMAPSSDataset(
    X_dev[train_mask],
    Y_dev[train_mask],
    unit_dev[train_mask],
    WINDOW_SIZE
)


val_dataset = NCMAPSSDataset(
    X_dev[val_mask],
    Y_dev[val_mask],
    unit_dev[val_mask],
    WINDOW_SIZE
)


test_dataset = NCMAPSSDataset(
    X_test,
    Y_test,
    unit_test,
    WINDOW_SIZE
)


print("\nDataset:")

print(
    f"Train windows : "
    f"{len(train_dataset):,}"
)

print(
    f"Val windows   : "
    f"{len(val_dataset):,}"
)

print(
    f"Test windows  : "
    f"{len(test_dataset):,}"
)


# ================================================================
# 9. DATALOADER
# ================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ================================================================
# 10. POSITIONAL ENCODING
# ================================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=500
    ):

        super().__init__()

        position = torch.arange(
            max_len
        ).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            *
            (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )


    def forward(self, x):

        return x + self.pe[:, :x.size(1)]


# ================================================================
# 11. CHANNEL ATTENTION
# ================================================================

class ChannelAttention(nn.Module):

    def __init__(
        self,
        channels,
        reduction=4
    ):

        super().__init__()

        hidden = max(
            channels // reduction,
            8
        )

        self.net = nn.Sequential(

            nn.Linear(
                channels,
                hidden
            ),

            nn.GELU(),

            nn.Linear(
                hidden,
                channels
            ),

            nn.Sigmoid()
        )


    def forward(self, x):

        # x: [B, C]

        weight = self.net(x)

        return x * weight


# ================================================================
# 12. CNN BRANCH
# ================================================================

class CNNBranch(nn.Module):

    def __init__(
        self,
        input_dim=18,
        d_model=96
    ):

        super().__init__()


        self.conv = nn.Sequential(

            nn.Conv1d(
                input_dim,
                64,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(64),

            nn.GELU(),

            nn.Conv1d(
                64,
                96,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(96),

            nn.GELU(),

            nn.Conv1d(
                96,
                d_model,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(d_model),

            nn.GELU()
        )


        self.attention = ChannelAttention(
            d_model
        )


        self.norm = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        # x:
        # [B, T, F]

        x = x.transpose(
            1,
            2
        )

        # [B, F, T]

        x = self.conv(x)

        # [B, D, T]

        x = x.transpose(
            1,
            2
        )

        # [B, T, D]

        # Global average pooling
        x = x.mean(dim=1)

        # [B, D]

        x = self.attention(x)

        x = self.norm(x)

        return x


# ================================================================
# 13. TRANSFORMER BRANCH
# ================================================================

class TransformerBranch(nn.Module):

    def __init__(
        self,
        input_dim=18,
        d_model=96,
        nhead=8,
        num_layers=2,
        dropout=0.15
    ):

        super().__init__()


        self.input_projection = nn.Linear(
            input_dim,
            d_model
        )


        self.position = PositionalEncoding(
            d_model,
            max_len=WINDOW_SIZE
        )


        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=d_model * 4,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )


        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )


        self.norm = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        # [B, T, F]

        x = self.input_projection(x)

        x = self.position(x)

        x = self.encoder(x)

        # 使用最后时刻
        # + 全局平均
        #
        # 比单独 global average
        # 保留更多退化末端信息

        last_feature = x[:, -1, :]

        mean_feature = x.mean(
            dim=1
        )

        x = 0.7 * last_feature + \
            0.3 * mean_feature

        x = self.norm(x)

        return x


# ================================================================
# 14. DYNAMIC GATED FUSION
# ================================================================

class DynamicGatedFusion(nn.Module):

    def __init__(
        self,
        d_model=96
    ):

        super().__init__()


        # 学习 CNN / Transformer 权重

        self.gate = nn.Sequential(

            nn.Linear(
                d_model * 2,
                d_model
            ),

            nn.LayerNorm(
                d_model
            ),

            nn.GELU(),

            nn.Linear(
                d_model,
                d_model
            ),

            nn.Sigmoid()
        )


        # 融合后的非线性变换

        self.transform = nn.Sequential(

            nn.Linear(
                d_model,
                d_model
            ),

            nn.LayerNorm(
                d_model
            ),

            nn.GELU(),

            nn.Dropout(
                DROPOUT
            )
        )


    def forward(
        self,
        cnn_feature,
        transformer_feature
    ):

        combined = torch.cat(
            [
                cnn_feature,
                transformer_feature
            ],
            dim=-1
        )

        gate = self.gate(
            combined
        )

        # 互补融合

        fused = (
            gate * cnn_feature
            +
            (1.0 - gate)
            * transformer_feature
        )

        fused = self.transform(
            fused
        )

        # residual
        fused = fused + 0.5 * (
            cnn_feature
            +
            transformer_feature
        )

        return fused, gate


# ================================================================
# 15. COMPLETE MODEL
# ================================================================

class CNNTransformerFusionV2(
    nn.Module
):

    def __init__(
        self,
        input_dim=18,
        d_model=96,
        nhead=8,
        num_layers=2
    ):

        super().__init__()


        self.cnn = CNNBranch(
            input_dim=input_dim,
            d_model=d_model
        )


        self.transformer = TransformerBranch(
            input_dim=input_dim,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers
        )


        self.fusion = DynamicGatedFusion(
            d_model=d_model
        )


        # 主预测头

        self.main_head = nn.Sequential(

            nn.Linear(
                d_model,
                64
            ),

            nn.LayerNorm(64),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(
                64,
                1
            )
        )


        # CNN auxiliary head

        self.cnn_head = nn.Sequential(

            nn.Linear(
                d_model,
                32
            ),

            nn.GELU(),

            nn.Linear(
                32,
                1
            )
        )


        # Transformer auxiliary head

        self.transformer_head = nn.Sequential(

            nn.Linear(
                d_model,
                32
            ),

            nn.GELU(),

            nn.Linear(
                32,
                1
            )
        )


    def forward(
        self,
        x,
        return_aux=False
    ):

        cnn_feature = self.cnn(x)

        transformer_feature = self.transformer(x)

        fused, gate = self.fusion(
            cnn_feature,
            transformer_feature
        )

        prediction = self.main_head(
            fused
        )

        if return_aux:

            cnn_pred = self.cnn_head(
                cnn_feature
            )

            transformer_pred = (
                self.transformer_head(
                    transformer_feature
                )
            )

            return (
                prediction,
                cnn_pred,
                transformer_pred,
                gate
            )

        return prediction


# ================================================================
# 16. MODEL
# ================================================================

model = CNNTransformerFusionV2(
    input_dim=18,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_TRANSFORMER_LAYERS
).to(device)


total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("\nModel parameters:")

print(
    f"{total_params:,}"
)


# ================================================================
# 17. LOSS / OPTIMIZER
# ================================================================

criterion = nn.MSELoss()


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )
)


scaler = torch.cuda.amp.GradScaler(
    enabled=USE_AMP
)


# ================================================================
# 18. TRAIN FUNCTION
# ================================================================

def train_one_epoch():

    model.train()

    total_loss = 0.0

    total_samples = 0


    pbar = tqdm(
        train_loader,
        desc="Training",
        ncols=100
    )


    for x, y in pbar:

        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        ).view(-1, 1)


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.cuda.amp.autocast(
            enabled=USE_AMP
        ):

            pred, cnn_pred, trans_pred, _ = model(
                x,
                return_aux=True
            )


            main_loss = criterion(
                pred,
                y
            )


            cnn_loss = criterion(
                cnn_pred,
                y
            )


            transformer_loss = criterion(
                trans_pred,
                y
            )


            loss = (
                main_loss
                +
                CNN_AUX_WEIGHT * cnn_loss
                +
                TRANSFORMER_AUX_WEIGHT
                * transformer_loss
            )


        scaler.scale(
            loss
        ).backward()


        scaler.unscale_(
            optimizer
        )


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )


        scaler.step(
            optimizer
        )

        scaler.update()


        batch_size = x.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size


        pbar.set_postfix(
            loss=f"{loss.item():.3f}"
        )


    return total_loss / total_samples


# ================================================================
# 19. VALIDATION
# ================================================================

def evaluate(loader):

    model.eval()

    predictions = []

    targets = []


    with torch.no_grad():

        for x, y in loader:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            ).view(-1, 1)


            with torch.cuda.amp.autocast(
                enabled=USE_AMP
            ):

                pred = model(x)


            predictions.append(
                pred.float().cpu().numpy()
            )

            targets.append(
                y.float().cpu().numpy()
            )


    predictions = np.concatenate(
        predictions
    ).reshape(-1)

    targets = np.concatenate(
        targets
    ).reshape(-1)


    mse = np.mean(
        (predictions - targets) ** 2
    )


    rmse = np.sqrt(mse)


    mae = np.mean(
        np.abs(
            predictions - targets
        )
    )


    return (
        mse,
        rmse,
        mae,
        predictions,
        targets
    )


# ================================================================
# 20. TRAINING
# ================================================================

best_val_rmse = float("inf")

best_epoch = 0

patience_counter = 0

train_history = []

val_history = []


print("\n" + "=" * 70)

print("START TRAINING")

print("=" * 70)


start_time = time.time()


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()


    train_loss = train_one_epoch()


    (
        val_mse,
        val_rmse,
        val_mae,
        _,
        _
    ) = evaluate(
        val_loader
    )


    scheduler.step(
        val_rmse
    )


    train_history.append(
        train_loss
    )

    val_history.append(
        val_rmse
    )


    epoch_time = (
        time.time()
        - epoch_start
    )


    print("\n" + "-" * 70)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS}"
    )

    print(
        f"Train MSE : "
        f"{train_loss:.6f}"
    )

    print(
        f"Val RMSE  : "
        f"{val_rmse:.6f}"
    )

    print(
        f"Val MAE   : "
        f"{val_mae:.6f}"
    )

    print(
        f"LR        : "
        f"{optimizer.param_groups[0]['lr']:.2e}"
    )

    print(
        f"Time      : "
        f"{epoch_time:.1f}s"
    )


    if val_rmse < best_val_rmse:

        best_val_rmse = val_rmse

        best_epoch = epoch

        patience_counter = 0


        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "epoch":
                    epoch,

                "val_rmse":
                    val_rmse,

                "val_mae":
                    val_mae,

                "X_min":
                    X_min,

                "X_max":
                    X_max,

                "config": {
                    "input_dim": 18,
                    "window_size":
                        WINDOW_SIZE,
                    "d_model":
                        D_MODEL,
                    "nhead":
                        NHEAD,
                    "num_layers":
                        NUM_TRANSFORMER_LAYERS
                }
            },
            MODEL_FILE
        )


        print(
            "  -> Best checkpoint saved."
        )


    else:

        patience_counter += 1

        print(
            f"  -> Early stopping "
            f"{patience_counter}/{PATIENCE}"
        )


        if patience_counter >= PATIENCE:

            print(
                "\nEarly stopping."
            )

            break


total_time = (
    time.time()
    - start_time
)


print("\nTraining finished.")

print(
    f"Total time : "
    f"{total_time / 60:.2f} min"
)


# ================================================================
# 21. LOAD BEST MODEL
# ================================================================

print("\n" + "=" * 70)

print("Loading best checkpoint...")

print("=" * 70)


checkpoint = torch.load(
    MODEL_FILE,
    map_location=device
)


model.load_state_dict(
    checkpoint["model_state_dict"]
)


print(
    f"Best epoch : "
    f"{checkpoint['epoch']}"
)

print(
    f"Best Val RMSE : "
    f"{checkpoint['val_rmse']:.6f}"
)


# ================================================================
# 22. TEST
# ================================================================

print("\n" + "=" * 70)

print("FINAL TEST")

print("=" * 70)


(
    test_mse,
    test_rmse,
    test_mae,
    predictions,
    targets
) = evaluate(
    test_loader
)


print(
    f"Test MSE  : "
    f"{test_mse:.6f}"
)

print(
    f"Test RMSE : "
    f"{test_rmse:.6f}"
)

print(
    f"Test MAE  : "
    f"{test_mae:.6f}"
)


# ================================================================
# 23. TEST WINDOW ENGINE LABELS
# ================================================================

test_units_for_windows = []


for idx in test_dataset.valid_indices:

    test_units_for_windows.append(
        unit_test[idx]
    )


test_units_for_windows = np.asarray(
    test_units_for_windows
)


# ================================================================
# 24. ENGINE LEVEL METRICS
# ================================================================

engine_results = []


print("\n" + "=" * 70)

print("ENGINE LEVEL RESULTS")

print("=" * 70)


for unit in np.unique(
    test_units_for_windows
):

    mask = (
        test_units_for_windows
        == unit
    )


    p = predictions[mask]

    t = targets[mask]


    mse = np.mean(
        (p - t) ** 2
    )


    rmse = np.sqrt(mse)


    mae = np.mean(
        np.abs(p - t)
    )


    engine_results.append(
        {
            "Engine":
                int(unit),

            "Samples":
                int(mask.sum()),

            "MSE":
                float(mse),

            "RMSE":
                float(rmse),

            "MAE":
                float(mae)
        }
    )


    print(
        f"Unit {int(unit):3d}: "
        f"RMSE={rmse:.4f}, "
        f"MAE={mae:.4f}, "
        f"N={mask.sum():,}"
    )


engine_df = pd.DataFrame(
    engine_results
)


engine_df.to_csv(
    ENGINE_CSV,
    index=False
)


# ================================================================
# 25. SAVE PREDICTIONS
# ================================================================

prediction_df = pd.DataFrame(
    {
        "Index":
            np.arange(
                len(predictions)
            ),

        "Engine":
            test_units_for_windows,

        "True_RUL":
            targets,

        "Pred_RUL":
            predictions,

        "Error":
            predictions - targets,

        "Abs_Error":
            np.abs(
                predictions - targets
            )
    }
)


prediction_df.to_csv(
    RESULT_CSV,
    index=False
)


# ================================================================
# 26. PREDICTION STATISTICS
# ================================================================

print("\n" + "=" * 70)

print("PREDICTION STATISTICS")

print("=" * 70)


print(
    f"True RUL mean : "
    f"{targets.mean():.4f}"
)

print(
    f"Pred RUL mean : "
    f"{predictions.mean():.4f}"
)

print(
    f"True RUL min  : "
    f"{targets.min():.4f}"
)

print(
    f"True RUL max  : "
    f"{targets.max():.4f}"
)

print(
    f"Pred RUL min  : "
    f"{predictions.min():.4f}"
)

print(
    f"Pred RUL max  : "
    f"{predictions.max():.4f}"
)


# ================================================================
# 27. PLOT
# ================================================================

fig = plt.figure(
    figsize=(14, 10)
)


# -------------------------------
# Training curve
# -------------------------------

ax1 = plt.subplot(
    2,
    2,
    1
)


ax1.plot(
    train_history,
    label="Train MSE"
)


ax1.plot(
    val_history,
    label="Val RMSE"
)


ax1.set_xlabel(
    "Epoch"
)

ax1.set_ylabel(
    "Metric"
)

ax1.set_title(
    "Training Curve"
)

ax1.legend()

ax1.grid(
    alpha=0.3
)


# -------------------------------
# Prediction vs True
# -------------------------------

ax2 = plt.subplot(
    2,
    2,
    2
)


sample_size = min(
    5000,
    len(predictions)
)


rng = np.random.default_rng(
    SEED
)


sample_idx = rng.choice(
    len(predictions),
    sample_size,
    replace=False
)


ax2.scatter(
    targets[sample_idx],
    predictions[sample_idx],
    s=4,
    alpha=0.4
)


min_v = min(
    targets.min(),
    predictions.min()
)


max_v = max(
    targets.max(),
    predictions.max()
)


ax2.plot(
    [min_v, max_v],
    [min_v, max_v],
    "--"
)


ax2.set_xlabel(
    "True RUL"
)

ax2.set_ylabel(
    "Predicted RUL"
)

ax2.set_title(
    f"Prediction vs True\n"
    f"RMSE={test_rmse:.4f}"
)

ax2.grid(
    alpha=0.3
)


# -------------------------------
# Residual
# -------------------------------

ax3 = plt.subplot(
    2,
    2,
    3
)


errors = (
    predictions
    - targets
)


ax3.hist(
    errors,
    bins=60,
    alpha=0.75
)


ax3.axvline(
    0,
    linestyle="--"
)


ax3.set_xlabel(
    "Prediction Error"
)

ax3.set_ylabel(
    "Count"
)

ax3.set_title(
    "Residual Distribution"
)

ax3.grid(
    alpha=0.3
)


# -------------------------------
# Engine RMSE
# -------------------------------

ax4 = plt.subplot(
    2,
    2,
    4
)


ax4.bar(
    engine_df["Engine"].astype(str),
    engine_df["RMSE"]
)


ax4.set_xlabel(
    "Engine"
)

ax4.set_ylabel(
    "RMSE"
)

ax4.set_title(
    "Engine-level RMSE"
)

ax4.grid(
    axis="y",
    alpha=0.3
)


plt.tight_layout()


plt.savefig(
    FIG_FILE,
    dpi=180
)


plt.show()


# ================================================================
# 28. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)

print(
    "FINAL SUMMARY | "
    "CNN + TRANSFORMER FUSION V2"
)

print("=" * 70)

print(
    f"Best Epoch : "
    f"{best_epoch}"
)

print(
    f"Val RMSE   : "
    f"{best_val_rmse:.6f}"
)

print(
    f"Test MSE   : "
    f"{test_mse:.6f}"
)

print(
    f"Test RMSE  : "
    f"{test_rmse:.6f}"
)

print(
    f"Test MAE   : "
    f"{test_mae:.6f}"
)

print()

print(
    f"Checkpoint : "
    f"{os.path.abspath(MODEL_FILE)}"
)

print(
    f"Predictions: "
    f"{os.path.abspath(RESULT_CSV)}"
)

print(
    f"Engine CSV : "
    f"{os.path.abspath(ENGINE_CSV)}"
)

print(
    f"Figure     : "
    f"{os.path.abspath(FIG_FILE)}"
)

print("=" * 70)

N-CMAPSS DS02-006 | CNN + TRANSFORMER FUSION V2
Device       : cuda
GPU          : NVIDIA GeForce RTX 3050 Laptop GPU
Window       : 50
Batch        : 512
d_model      : 96
Transformer  : 2 layers
AMP          : True

Loading N-CMAPSS data...
X_dev  : (5263447, 18)
X_test : (1253743, 18)
Y_dev  : (5263447,)
Y_test : (1253743,)
Dev engines  : [ 2  5 10 16 18 20]
Test engines : [11 14 15]

ENGINE SPLIT
Train engines : [ 2  5 10 16 18]
Validation    : 20

Normalization:
Scaler fitted ONLY on training engines.


e:\RUL\venv\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
C:\Users\Alina\AppData\Local\Temp\ipykernel_12056\1123813932.py:1018: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(



Dataset:
Train windows : 4,495,042
Val windows   : 768,111
Test windows  : 1,253,596

Model parameters:
345,723

START TRAINING


Training:   0%|                                                            | 0/8779 [00:00<?, ?it/s]C:\Users\Alina\AppData\Local\Temp\ipykernel_12056\1123813932.py:1061: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
Training: 100%|████████████████████████████████████| 8779/8779 [10:54<00:00, 13.41it/s, loss=83.319]
C:\Users\Alina\AppData\Local\Temp\ipykernel_12056\1123813932.py:1168: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



----------------------------------------------------------------------
Epoch 01/30
Train MSE : 419.752886
Val RMSE  : 8.908136
Val MAE   : 6.632316
LR        : 3.00e-04
Time      : 709.7s
  -> Best checkpoint saved.


Training: 100%|████████████████████████████████████| 8779/8779 [13:16<00:00, 11.03it/s, loss=68.592]


... (6 outputs left out)


----------------------------------------------------------------------
Epoch 05/30
Train MSE : 42.994898
Val RMSE  : 11.892262
Val MAE   : 8.938645
LR        : 3.00e-04
Time      : 859.7s
  -> Early stopping 3/7


Training: 100%|████████████████████████████████████| 8779/8779 [13:31<00:00, 10.81it/s, loss=35.991]



----------------------------------------------------------------------
Epoch 06/30
Train MSE : 39.215429
Val RMSE  : 9.892253
Val MAE   : 7.439876
LR        : 1.50e-04
Time      : 867.5s
  -> Early stopping 4/7


Training:  10%|███▊                                 | 915/8779 [01:21<08:37, 15.21it/s, loss=32.457]